# Course 8: Research Engineering Interviews
## Session 3: Coding II, tensors

**Goal:** solve ten tensor exercises without Python loops, predicting every shape.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** the tests and reference solutions here were run once on a CPU (November 2026). Cells marked TODO are stubs for you; try them under a timer before opening the reference solutions at the bottom.

### Warm-up: predict each shape before running

In [ ]:
import torch
A, B = torch.randn(4, 3, 5), torch.randn(4, 5, 2)
C = torch.einsum("bij,bjk->bik", A, B)                 # (4, 3, 2)
print(C.shape, torch.allclose(C, A @ B))
x, y = torch.randn(6, 3), torch.randn(4, 3)
d2 = ((x[:, None, :] - y[None, :, :]) ** 2).sum(-1)    # (6, 4)
print(d2.shape, torch.allclose(d2.sqrt(), torch.cdist(x, y), atol=1e-5))
print(torch.broadcast_shapes((5, 1, 4), (3, 1)))

### The ten exercises (no Python loops)
1. `row_normalize(x)`: each row sums to 1.
2. `one_hot(labels, n)`.
3. `pairwise_sq_dist(x, y)`: (n, m).
4. `cosine_sim(x)`: (n, n).
5. `causal_mask(T)`: (T, T) of 1s on and below the diagonal.
6. `batched_matmul(A, B)` with einsum.
7. `true_class_logprob(logits, labels)`: (n,).
8. `top3(x)`: indices of the 3 largest per row.
9. `moving_average(x, k)`: along the last axis, windows of k.
10. `masked_mean(x, mask)`: mean over time of (B, T, D) where mask (B, T) is 1 for real tokens.

In [ ]:
import torch.nn.functional as F
def run_tests(row_normalize, one_hot, pairwise_sq_dist, cosine_sim, causal_mask, batched_matmul,
              true_class_logprob, top3, moving_average, masked_mean):
    torch.manual_seed(0)
    x = torch.rand(5, 7)
    assert torch.allclose(row_normalize(x).sum(1), torch.ones(5))
    lab = torch.tensor([0, 2, 1])
    assert torch.equal(one_hot(lab, 3), F.one_hot(lab, 3).float())
    a, b = torch.randn(6, 3), torch.randn(4, 3)
    assert torch.allclose(pairwise_sq_dist(a, b), torch.cdist(a, b) ** 2, atol=1e-4)
    cs = cosine_sim(a)
    assert cs.shape == (6, 6) and torch.allclose(cs.diag(), torch.ones(6), atol=1e-5)
    assert torch.equal(causal_mask(4), torch.tril(torch.ones(4, 4)))
    A, B = torch.randn(4, 3, 5), torch.randn(4, 5, 2)
    assert torch.allclose(batched_matmul(A, B), A @ B, atol=1e-5)
    logits, labels = torch.randn(8, 10), torch.randint(0, 10, (8,))
    assert torch.allclose(true_class_logprob(logits, labels).mean(), -F.cross_entropy(logits, labels), atol=1e-5)
    assert torch.equal(top3(x), x.topk(3, dim=1).indices)
    s = torch.arange(6.0)[None, :]
    assert torch.allclose(moving_average(s, 3), torch.tensor([[1.0, 2.0, 3.0, 4.0]]))
    h = torch.randn(2, 4, 3); m = torch.tensor([[1, 1, 0, 0], [1, 1, 1, 1]])
    assert torch.allclose(masked_mean(h, m)[0], h[0, :2].mean(0), atol=1e-6)
    print("all ten pass")

In [ ]:
def row_normalize(x): raise NotImplementedError             # TODO, and so on for all ten
# run_tests(row_normalize, one_hot, pairwise_sq_dist, cosine_sim, causal_mask, batched_matmul,
#           true_class_logprob, top3, moving_average, masked_mean)

### A silent broadcasting bug
The wrong line runs without error and gives wrong values.

In [ ]:
x = torch.rand(3, 3)
wrong = x / x.sum(dim=1)                    # divides column j by the sum of row j
right = x / x.sum(dim=1, keepdim=True)
print("row sums, wrong:", wrong.sum(1), "| right:", right.sum(1))

---
## Reference solutions
Try every problem under the timer first. Then run this section: it defines working versions and runs the same tests.

In [ ]:
def row_normalize(x): return x / x.sum(dim=1, keepdim=True)                      # (n, d)
def one_hot(labels, n): return (labels[:, None] == torch.arange(n)[None, :]).float()   # (n, classes)
def pairwise_sq_dist(x, y): return ((x[:, None, :] - y[None, :, :]) ** 2).sum(-1)  # (n, m)
def cosine_sim(x):
    u = x / x.norm(dim=1, keepdim=True)                                            # (n, d)
    return u @ u.T                                                                 # (n, n)
def causal_mask(T): return torch.tril(torch.ones(T, T))
def batched_matmul(A, B): return torch.einsum("bij,bjk->bik", A, B)
def true_class_logprob(logits, labels):
    return torch.log_softmax(logits, dim=-1)[torch.arange(len(labels)), labels]    # (n,)
def top3(x): return x.topk(3, dim=1).indices
def moving_average(x, k): return x.unfold(-1, k, 1).mean(-1)
def masked_mean(x, mask):
    m = mask[:, :, None].float()                                                   # (B, T, 1)
    return (x * m).sum(1) / m.sum(1)                                               # (B, D)

run_tests(row_normalize, one_hot, pairwise_sq_dist, cosine_sim, causal_mask, batched_matmul,
          true_class_logprob, top3, moving_average, masked_mean)